In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/baostock')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
code = codes[0]
dt = datasource.get_data(code)
dt.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST
date,,,,,,,,,,,,,
2010-01-04,sh.600000,5.079055,5.088362,4.923170,4.930150,5.046482,66191338,1.419984e+09,2,0.835129,1,-2.3052,0
2010-01-05,sh.600000,4.981336,5.020889,4.837085,4.967376,4.930150,115147943,2.436891e+09,2,1.452808,1,0.7551,0
2010-01-06,sh.600000,4.953417,4.955743,4.858024,4.869658,4.967376,96782575,2.034174e+09,2,1.221095,1,-1.9672,0
2010-01-07,sh.600000,4.858024,4.895251,4.723079,4.760305,4.869658,85236072,1.761801e+09,2,1.075414,1,-2.2456,0
2010-01-08,sh.600000,4.732386,4.839411,4.723079,4.813818,4.760305,65707646,1.349532e+09,2,0.829026,1,1.1241,0


# 在少于0的时候金叉算

In [2]:
# macd的3个参数
fastperiod=12
slowperiod=26
signalperiod=9
# 如下是macd，分别是dif， signal(dea)，hist
dt['macd_dif'], dt['macd_signal'], dt['macdhist'] = talib.MACD(dt['close'], fastperiod=fastperiod, slowperiod=slowperiod, signalperiod=signalperiod)
dt['pre_macd_dif'] = dt['macd_dif'].shift(1)
dt['pre_macd_signal'] = dt['macd_signal'].shift(1)
# 未来几天的收盘价
for i in range(5):
    dt[f'nextClose{i+1}'] = dt['close'].shift(-(i+1))
dt.iloc[50:60].head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,...,macd_dif,macd_signal,macdhist,pre_macd_dif,pre_macd_signal,nextClose1,nextClose2,nextClose3,nextClose4,nextClose5
date,,,,,,,,,,,,,,,,,,,,,
2010-04-06,sh.600000,5.362905,5.439684,5.330332,5.348945,5.330332,53809591,1.243972e+09,2,0.678909,...,0.159085,0.129941,0.029144,0.155930,0.122655,5.300086,5.253553,5.337312,5.218654,5.358252
2010-04-07,sh.600000,5.351272,5.369885,5.244247,5.300086,5.348945,50601633,1.150564e+09,2,0.638435,...,0.155846,0.135122,0.020724,0.159085,0.129941,5.253553,5.337312,5.218654,5.358252,5.323352
2010-04-08,sh.600000,5.234940,5.344292,5.234940,5.253553,5.300086,46876305,1.063341e+09,2,0.591433,...,0.147820,0.137662,0.010159,0.155846,0.135122,5.337312,5.218654,5.358252,5.323352,5.297759
2010-04-09,sh.600000,5.258206,5.355925,5.255880,5.337312,5.253553,38392741,8.766580e+08,2,0.484397,...,0.146530,0.139435,0.007094,0.147820,0.137662,5.218654,5.358252,5.323352,5.297759,5.132568
2010-04-12,sh.600000,5.351272,5.362905,5.167467,5.218654,5.337312,57895624,1.306221e+09,2,0.730462,...,0.134383,0.138425,-0.004042,0.146530,0.139435,5.358252,5.323352,5.297759,5.132568,4.783572


In [3]:
dt2 = dt.loc[(dt['macd_dif'] < 0 ) & (dt['macd_signal'] < 0 ) & (dt['macd_dif'] > dt['macd_signal'] ) & (dt['pre_macd_dif'] < dt['pre_macd_signal'])].dropna()

In [4]:
print(len(dt2))
dt2.tail()

97


,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,...,macd_dif,macd_signal,macdhist,pre_macd_dif,pre_macd_signal,nextClose1,nextClose2,nextClose3,nextClose4,nextClose5
date,,,,,,,,,,,,,,,,,,,,,
2024-12-23,sh.600000,9.346560,9.783315,9.317443,9.637730,9.298032,116444693,1.156091e+09,2,0.3967,...,-0.001934,-0.020062,0.018129,-0.029019,-0.024594,9.831844,10.045368,10.035663,10.055074,10.161836
2025-03-14,sh.600000,9.977429,10.181248,9.938606,10.113308,9.919195,78422138,8.151089e+08,2,0.2672,...,-0.008296,-0.018104,0.009809,-0.029492,-0.020556,10.288010,10.326833,10.336539,10.346244,10.161836
2025-04-16,sh.600000,10.142425,10.288010,10.045368,10.288010,10.161836,74094520,7.773786e+08,2,0.2524,...,-0.012293,-0.026392,0.014099,-0.043336,-0.029917,10.433595,10.491829,10.258893,10.394773,10.355950
2025-10-14,sh.600000,12.450000,12.870000,12.310000,12.770000,12.510000,146399373,1.860042e+09,2,0.4600,...,-0.352854,-0.359337,0.006482,-0.415555,-0.360957,13.170000,13.320000,13.320000,12.990000,12.990000
2025-11-19,sh.600000,11.430000,11.620000,11.400000,11.560000,11.450000,93264386,1.078378e+09,2,0.2800,...,-0.310230,-0.316614,0.006384,-0.324330,-0.318210,11.700000,11.520000,11.530000,11.610000,11.480000


In [8]:
# 这里看看比值
for i in range(5):
    dt2[f'rate{i+1}'] = dt2[f'nextClose{i+1}']/dt2['close']
    # 这里看一下结果把
    _avg = dt2[f'rate{i+1}'].mean()
    _median = dt2[f'rate{i+1}'].median()
    print(f'{i+1}平均值:{_avg}, 中位值:{_median}')

1平均值:1.0010632573114437, 中位值:1.0
2平均值:1.0011717081272575, 中位值:0.99812734082397
3平均值:1.0013377517794269, 中位值:0.9978118161925602
4平均值:1.004087518536959, 中位值:0.997828447339848
5平均值:1.0039529720883902, 中位值:0.9987577639751554


看起来像是不变啊

# 大于0时候的金叉

In [9]:
dt3 = dt.loc[(dt['macd_dif'] > 0 ) & (dt['macd_signal'] > 0 ) & (dt['macd_dif'] > dt['macd_signal'] ) & (dt['pre_macd_dif'] < dt['pre_macd_signal'])].dropna()
print(len(dt3))
dt3.tail()

47


,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,...,macd_dif,macd_signal,macdhist,pre_macd_dif,pre_macd_signal,nextClose1,nextClose2,nextClose3,nextClose4,nextClose5
date,,,,,,,,,,,,,,,,,,,,,
2025-01-27,sh.600000,10.113308,10.385067,10.093897,10.336539,10.084191,59488208,6.326946e+08,2,0.2027,...,0.121420,0.105619,0.015801,0.097064,0.101669,10.103602,10.123014,10.035663,9.967723,9.996840
2025-06-20,sh.600000,12.403846,12.636782,12.355318,12.636782,12.365024,69793426,9.034286e+08,2,0.2378,...,0.281674,0.272655,0.009019,0.259904,0.270400,12.918247,12.918247,13.063832,13.219123,13.151183
2025-08-06,sh.600000,13.750000,13.850000,13.630000,13.790000,13.750000,84891798,1.170589e+09,2,0.2805,...,0.074901,0.062218,0.012683,0.029489,0.059048,13.930000,14.170000,13.980000,13.890000,13.830000
2025-08-25,sh.600000,13.900000,14.050000,13.760000,14.030000,13.940000,63137208,8.771624e+08,2,0.2086,...,0.169390,0.161817,0.007573,0.157350,0.159923,14.030000,13.750000,13.860000,13.630000,13.490000
2025-09-11,sh.600000,13.930000,14.170000,13.880000,14.120000,13.970000,83259271,1.168876e+09,2,0.2730,...,0.101100,0.089227,0.011874,0.081131,0.086258,13.600000,13.430000,13.160000,12.960000,12.750000


In [10]:
# 这里看看比值
for i in range(5):
    dt3[f'rate{i+1}'] = dt3[f'nextClose{i+1}']/dt3['close']
    # 这里看一下结果把
    _avg = dt3[f'rate{i+1}'].mean()
    _median = dt3[f'rate{i+1}'].median()
    _std =  dt3[f'rate{i+1}'].std()
    print(f'{i+1}平均值:{_avg}, 中位值:{_median},标准差:{_std}')

1平均值:0.9983037893473553, 中位值:0.9988938053097345,标准差:0.01751459152937467
2平均值:0.9985697963563479, 中位值:0.996652719665272,标准差:0.02240833340083273
3平均值:1.0017154379318691, 中位值:0.9955456570155902,标准差:0.03616515108020498
4平均值:1.0024949822582812, 中位值:0.9989754098360656,标准差:0.041176235433468135
5平均值:0.997829564524047, 中位值:0.9944320712694876,标准差:0.04306979374084573
